In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
MODEL_NAME = 'all-mpnet-base-v2'
TOP_K      = 5
THRESHOLD  = 0.3

# Load video index
df        = pd.read_csv('video_index.csv')
meta_cols = ['video_id', 'title', 'datetime', 'transcript']
emb_cols  = [col for col in df.columns if col.startswith('emb_')]

meta       = df[meta_cols].copy()
embeddings = df[emb_cols].values

# Load model
model = SentenceTransformer(MODEL_NAME)

print(f"Model     : {MODEL_NAME}")
print(f"Top-K     : {TOP_K}")
print(f"Threshold : {THRESHOLD}")
print(f"Videos    : {len(meta)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model     : all-mpnet-base-v2
Top-K     : 5
Threshold : 0.3
Videos    : 123


In [3]:
def returnSearchResults(query, top_k=TOP_K, threshold=THRESHOLD):
    if not query.strip():
        return []

    # Encode query
    query_embedding = model.encode([query])

    # Compute cosine similarity
    scores = cosine_similarity(query_embedding, embeddings)[0]

    # Rank and filter
    ranked_all    = np.argsort(scores)[::-1]
    mask          = scores[ranked_all] >= threshold
    filtered      = ranked_all[mask][:top_k]

    if len(filtered) == 0:
        return []

    # Format results
    results = []
    for rank, idx in enumerate(filtered, 1):
        video_id = meta.iloc[idx]['video_id']
        title    = meta.iloc[idx]['title']
        date     = meta.iloc[idx]['datetime']
        score    = round(float(scores[idx]), 3)
        link     = f"https://www.youtube.com/watch?v={video_id}"

        results.append({
            'rank'    : rank,
            'title'   : title,
            'video_id': video_id,
            'date'    : date,
            'score'   : score,
            'link'    : link
        })

    return results

print("Search function ready!")

Search function ready!


In [4]:
query   = "What is dust?"
results = returnSearchResults(query)

if not results:
    print("No results found.")
else:
    print(f"Top {len(results)} results for: '{query}'\n")
    for r in results:
        print(f"Rank  : {r['rank']}")
        print(f"Title : {r['title']}")
        print(f"Score : {r['score']}")
        print(f"Link  : {r['link']}")
        print("-" * 50)

Top 3 results for: 'What is dust?'

Rank  : 1
Title : Is dust really people?
Score : 0.809
Link  : https://www.youtube.com/watch?v=2KaAHD2TOos
--------------------------------------------------
Rank  : 2
Title : The Secret Ingredient Hiding in Masters Golf Sand ⛳️
Score : 0.329
Link  : https://www.youtube.com/watch?v=iBkCJBK1N5g
--------------------------------------------------
Rank  : 3
Title : Why NASA Punched an Asteroid
Score : 0.301
Link  : https://www.youtube.com/watch?v=tp9BQ88rNso
--------------------------------------------------


In [5]:
test_queries = [
    "What is dust?",               # relevant query
    "How do birds fly?",           # relevant query
    "What is inside a black hole?",# relevant query
    "machine learning overfitting", # irrelevant — no ML videos
    "",                            # edge case — empty query
    "xyzabc123",                   # edge case — random text
]

for q in test_queries:
    results = returnSearchResults(q)
    print(f"Query   : '{q}'")
    print(f"Results : {len(results)}")
    if results:
        print(f"Top 1   : {results[0]['title']} (score: {results[0]['score']})")
    print("-" * 50)

Query   : 'What is dust?'
Results : 3
Top 1   : Is dust really people? (score: 0.809)
--------------------------------------------------
Query   : 'How do birds fly?'
Results : 5
Top 1   : Why do birds fly in a V? (score: 0.721)
--------------------------------------------------
Query   : 'What is inside a black hole?'
Results : 0
--------------------------------------------------
Query   : 'machine learning overfitting'
Results : 0
--------------------------------------------------
Query   : ''
Results : 0
--------------------------------------------------
Query   : 'xyzabc123'
Results : 0
--------------------------------------------------
